In [2]:
# Challenge 2: FAQ Chatbot

# This notebook imports FAQ data into BigQuery, generates embeddings for each question-and-answer pair, performs vector search, and uses Gemini to answer user questions.

In [3]:
# Project setup
PROJECT_ID = "qwiklabs-gcp-01-ca95818a69b6"
REGION = "us-central1"

DATASET_ID = "aurora_bay"
TABLE_ID = "faqs"
EMBEDDING_TABLE_ID = "faqs_with_embeddings"

GCS_URI = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

In [4]:
from google.cloud import bigquery
import pandas as pd

# BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)

print(f"Using project: {PROJECT_ID}")

Using project: qwiklabs-gcp-01-ca95818a69b6


In [5]:
from google.api_core.exceptions import Conflict

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "US"

try:
    bq_client.create_dataset(dataset_ref)
    print(f"Created dataset: {PROJECT_ID}.{DATASET_ID}")
except Conflict:
    print(f"Dataset already exists: {PROJECT_ID}.{DATASET_ID}")

Created dataset: qwiklabs-gcp-01-ca95818a69b6.aurora_bay


In [6]:
table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = bq_client.load_table_from_uri(
    GCS_URI,
    table_ref,
    job_config=job_config,
)

load_job.result()

table = bq_client.get_table(table_ref)
print(f"Loaded {table.num_rows} rows into {table_ref}")

Loaded 50 rows into qwiklabs-gcp-01-ca95818a69b6.aurora_bay.faqs


In [7]:
query = f"""
SELECT *
FROM `{table_ref}`
LIMIT 5
"""

df = bq_client.query(query).to_dataframe()
df

,string_field_0,string_field_1
0,When was Aurora Bay founded?,Aurora Bay was founded in 1901 by a group of f...
1,What is the population of Aurora Bay?,Aurora Bay has a population of approximately 3...
2,Where is the Aurora Bay Town Hall located?,The Town Hall is located at 100 Harbor View Ro...
3,Who is the current mayor of Aurora Bay?,"The current mayor is Linda Greenwood, elected ..."
4,What are the primary industries in Aurora Bay?,The primary industries include commercial fish...


In [8]:
table = bq_client.get_table(table_ref)

for schema_field in table.schema:
    print(schema_field.name, schema_field.field_type)

string_field_0 STRING
string_field_1 STRING


In [9]:
rename_query = f"""
CREATE OR REPLACE TABLE `{table_ref}` AS
SELECT
  string_field_0 AS question,
  string_field_1 AS answer
FROM `{table_ref}`
"""

bq_client.query(rename_query).result()

print(f"Renamed columns in {table_ref}")

Renamed columns in qwiklabs-gcp-01-ca95818a69b6.aurora_bay.faqs


In [10]:
df = bq_client.query(f"""
SELECT *
FROM `{table_ref}`
LIMIT 5
""").to_dataframe()

df

,question,answer
0,What is the procedure for trash collection?,Trash is collected once a week based on your a...
1,How can local residents receive alerts about e...,Residents can sign up for text or email alerts...
2,When are the town council meetings held?,Town council meetings are held every second Tu...
3,What cultural events are unique to Aurora Bay?,"Besides the Aurora Lights Winter Festival, the..."
4,How can I find local emergency shelters during...,Emergency shelters are typically set up at Aur...


In [11]:
import vertexai
from vertexai.language_models import TextEmbeddingModel

vertexai.init(project=PROJECT_ID, location=REGION)

embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")

print("Vertex AI initialized and embedding model loaded.")

Vertex AI initialized and embedding model loaded.


/usr/local/lib/python3.12/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [12]:
# Fetch data from BigQuery
query = f"""
SELECT question, answer
FROM `{table_ref}`
"""

df = bq_client.query(query).to_dataframe()

print(f"Loaded {len(df)} rows")
df.head()

Loaded 50 rows


,question,answer
0,What is the procedure for trash collection?,Trash is collected once a week based on your a...
1,How can local residents receive alerts about e...,Residents can sign up for text or email alerts...
2,When are the town council meetings held?,Town council meetings are held every second Tu...
3,What cultural events are unique to Aurora Bay?,"Besides the Aurora Lights Winter Festival, the..."
4,How can I find local emergency shelters during...,Emergency shelters are typically set up at Aur...


In [13]:
def get_embedding(text):
    return embedding_model.get_embeddings([text])[0].values

# Combine question + answer
df["combined_text"] = df["question"] + " " + df["answer"]

# Generate embeddings (this may take a minute)
df["embedding"] = df["combined_text"].apply(get_embedding)

df.head()

,question,answer,combined_text,embedding
0,What is the procedure for trash collection?,Trash is collected once a week based on your a...,What is the procedure for trash collection? Tr...,"[-0.0800468772649765, -0.04155504330992699, -0..."
1,How can local residents receive alerts about e...,Residents can sign up for text or email alerts...,How can local residents receive alerts about e...,"[-0.07066945731639862, -0.030102236196398735, ..."
2,When are the town council meetings held?,Town council meetings are held every second Tu...,When are the town council meetings held? Town ...,"[-0.034046899527311325, -0.08912285417318344, ..."
3,What cultural events are unique to Aurora Bay?,"Besides the Aurora Lights Winter Festival, the...",What cultural events are unique to Aurora Bay?...,"[-0.033465296030044556, 0.013448487035930157, ..."
4,How can I find local emergency shelters during...,Emergency shelters are typically set up at Aur...,How can I find local emergency shelters during...,"[-0.07826904952526093, 0.028559423983097076, -..."


In [14]:
embedding_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{EMBEDDING_TABLE_ID}"

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

load_job = bq_client.load_table_from_dataframe(
    df,
    embedding_table_ref,
    job_config=job_config,
)

load_job.result()

print(f"Saved embeddings to {embedding_table_ref}")

Saved embeddings to qwiklabs-gcp-01-ca95818a69b6.aurora_bay.faqs_with_embeddings


In [15]:
preview_df = bq_client.query(f"""
SELECT
  question,
  answer,
  ARRAY_LENGTH(embedding) AS embedding_dimensions
FROM `{embedding_table_ref}`
LIMIT 5
""").to_dataframe()

preview_df

,question,answer,embedding_dimensions
0,What is the procedure for trash collection?,Trash is collected once a week based on your a...,768
1,How can local residents receive alerts about e...,Residents can sign up for text or email alerts...,768
2,When are the town council meetings held?,Town council meetings are held every second Tu...,768
3,What cultural events are unique to Aurora Bay?,"Besides the Aurora Lights Winter Festival, the...",768
4,How can I find local emergency shelters during...,Emergency shelters are typically set up at Aur...,768


In [27]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two embedding vectors.
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def search_faqs(user_question, top_k=3):
    """
    Generate an embedding for the user's question and return the most similar FAQ rows.
    """
    query_embedding = get_embedding(user_question)

    search_df = df.copy()
    search_df["similarity"] = search_df["embedding"].apply(
        lambda emb: cosine_similarity(query_embedding, emb)
    )

    results = search_df.sort_values("similarity", ascending=False).head(top_k)

    return results[["question", "answer", "similarity"]]

In [17]:
search_faqs("What time is check-in?")

,question,answer,similarity
0,What is the procedure for trash collection?,Trash is collected once a week based on your a...,0.518352
36,Are there specific quiet hours or noise ordina...,Yes. Residential noise ordinances go into effe...,0.516571
20,When does the local fishermen’s market usually...,The fishermen’s market runs every Saturday fro...,0.469824


In [18]:
from vertexai.generative_models import GenerativeModel

gemini_model = GenerativeModel("gemini-2.5-flash")

print("Gemini model loaded.")

Gemini model loaded.


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [26]:
def generate_answer(user_question):
    """
    Uses vector search results + Gemini to generate a grounded response.
    """
    results = search_faqs(user_question, top_k=3)

    context = "\n\n".join(
        [f"Q: {row['question']}\nA: {row['answer']}" for _, row in results.iterrows()]
    )

    prompt = f"""
You are a helpful customer service assistant for Aurora Bay.

Use ONLY the information below to answer the question.

Context:
{context}

Question:
{user_question}

Answer:
"""
    response = gemini_model.generate_content(prompt)
    return response.text

In [21]:
generate_answer("What lodging options are available?")

'Options include the Aurora Bay Lodge (near the harbor), several bed-and-breakfasts, and a small hostel on Main Street.'

In [22]:
def chatbot():
    """
    Simple command-line chatbot loop.
    Type 'exit' to stop.
    """
    print("Aurora Bay FAQ Chatbot")
    print("Type 'exit' to quit.\n")

    while True:
        user_question = input("You: ")

        if user_question.lower() in ["exit", "quit", "q"]:
            print("Goodbye!")
            break

        answer = generate_answer(user_question)
        print(f"\nBot: {answer}\n")

In [25]:
chatbot()

Aurora Bay FAQ Chatbot
Type 'exit' to quit.

You: exit
Goodbye!


## Summary

This notebook completed the following:

1. Created a BigQuery dataset and table for the Aurora Bay FAQ CSV file.
2. Loaded FAQ data from Cloud Storage into BigQuery.
3. Renamed raw CSV columns to `question` and `answer`.
4. Generated embeddings for each combined question-and-answer pair using Vertex AI.
5. Stored the FAQ data and embeddings in a new BigQuery table.
6. Built a vector search function using cosine similarity.
7. Used Gemini to generate grounded answers from retrieved FAQ context.
8. Created an interactive chatbot loop.

The chatbot uses Retrieval-Augmented Generation (RAG): it first retrieves relevant FAQ entries using vector similarity, then passes those entries to Gemini to generate an accurate response.

# **High-probability working questions**



*  What lodging options are available?
*  Where can I stay in Aurora Bay?
*  What types of accommodations are there?

* What activities can I do in Aurora Bay?
* What can visitors do there?
* Are there outdoor activities available?

* How do I get to Aurora Bay?
* What transportation options are available?
* How can I travel to Aurora Bay?

* Are there restaurants in Aurora Bay?
* Where can I eat there?
* What dining options are available?

* What is Aurora Bay known for?
* Tell me about Aurora Bay.
* Why should I visit Aurora Bay?



# **High-probability not working questions**

* Do you offer late checkout?
* How much does it cost to stay at Aurora Bay Lodge?
* What is the weather in Aurora Bay today?